# 🧪 MODULE 40 — DEEP LEARNING EXPERIMENTATION

# PART 2 — TRAINING TECHNIQUES & EXPERIMENT TRACKING

---

# 40.2 Training Techniques

Training a deep-learning model is not simply:

```text
Define model
     ↓
model.fit()
     ↓
Done
```

In real projects, we need mechanisms to control and monitor training:

```text
                    TRAINING
                       │
        ┌──────────────┼──────────────┐
        ↓              ↓              ↓
 Learning-rate     Early          Checkpointing
 Scheduling        Stopping
        │              │              │
        └──────────────┼──────────────┘
                       ↓
                 Stable Training
                       ↑
                 Gradient Clipping
```

These techniques help us:

- train faster
- prevent overfitting
- recover from failures
- stabilize optimization
- save the best model
- understand what happened during training

---

# 1️⃣ Learning-Rate Scheduling

## What is Learning-Rate Scheduling?

Previously we learned that the learning rate controls the size of the optimization step:

$$
\theta_{t+1}
=
\theta_t
-
\eta_t\nabla J(\theta_t)
$$

Notice something important:

$$
\eta_t
$$

The learning rate can depend on the training step $t$.

Instead of keeping:

```text
learning rate = 0.001
```

for the entire training process, we can change it over time.

This is called **learning-rate scheduling**.

Learning-rate schedules define how the learning rate changes during training, commonly by epoch or iteration. [Apache MXNet](https://mxnet.apache.org/versions/1.4.1/tutorials/gluon/learning_rate_schedules.html)

---

## 🧠 Why Change the Learning Rate?

Imagine training starts here:

```text
Random parameters
       ↓
Large error
```

At the beginning, we generally want to make relatively large progress.

Later:

```text
Near good solution
       ↓
Fine adjustments needed
```

A learning rate that is too large near the end can cause the optimizer to jump around the minimum.

So conceptually:

```text
Early training
     ↓
Higher LR
     ↓
Fast movement

Later training
     ↓
Lower LR
     ↓
Fine adjustment
```

---

## 🖼️ Learning-Rate Schedule

### Step Decay

```markdown
![Step Learning Rate Decay](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/doc/tutorials/lr_schedules/factor.png)
```

![Step Learning Rate Decay](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/doc/tutorials/lr_schedules/factor.png)

The step schedule reduces the learning rate at predefined intervals. MXNet's documentation uses this as a standard example of stepwise decay. [Apache MXNet](https://mxnet.apache.org/versions/1.4.1/tutorials/gluon/learning_rate_schedules.html)

---

# Common Learning-Rate Schedules

There are several important strategies.

---

## 1. Step Decay

The learning rate is reduced at specific points.

For example:

```text
Epoch 1–10     → 0.001
Epoch 11–20    → 0.0001
Epoch 21–30    → 0.00001
```

Mathematically:

$$
\eta_t =
\begin{cases}
0.001 & t < 10\\
0.0001 & 10 \leq t < 20\\
0.00001 & t \geq 20
\end{cases}
$$

### Intuition

```text
Learning rate

0.001 ────────────┐
                  │
0.0001            └───────────┐
                              │
0.00001                       └────────────
       0        10        20       Epoch
```

---

# 2. Exponential Decay

The learning rate gradually decreases.

$$
\eta_t
=
\eta_0e^{-kt}
$$

Where:

- $\eta_0$ = initial learning rate
- $k$ = decay rate
- $t$ = training step

Instead of sudden drops:

```text
0.001
  │
  └───────
          ╲
           ╲
            ╲
             ╲
              0
```

the decrease is smoother.

---

# 3. Cosine Annealing

Another important schedule is **cosine annealing**.

```markdown
![Cosine Annealing Learning Rate](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/doc/tutorials/lr_schedules/cosine.png)
```

![Cosine Annealing Learning Rate](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/doc/tutorials/lr_schedules/cosine.png)

A cosine schedule smoothly decreases the learning rate according to a cosine-shaped curve. [Apache MXNet](https://mxnet.apache.org/versions/1.4.1/tutorials/gluon/learning_rate_schedules.html)

A simplified form is:

$$
\eta_t
=
\eta_{\min}
+
\frac{1}{2}
(\eta_{\max}-\eta_{\min})
\left(
1+\cos\left(\frac{\pi t}{T}\right)
\right)
$$

Where:

- $\eta_{\max}$ = maximum learning rate
- $\eta_{\min}$ = minimum learning rate
- $T$ = total schedule length

---

# 4. Learning-Rate Warmup

Warmup is particularly important in modern deep learning.

Instead of starting immediately with a large learning rate:

```text
0.001
```

we gradually increase it.

```text
LR

0.001                 ─────────────
                    /
                  /
                /
0.0001 ─────────
       │
       └──────────────────────
             Steps
```

For example:

```text
Step 1      → 0.00001
Step 100     → 0.0001
Step 500     → 0.0005
Step 1000    → 0.001
```

Then the normal schedule begins.

Google's Deep Learning Tuning Playbook specifically discusses warmup as a way of addressing early-training instability. [Google for Developers](https://developers.google.com/machine-learning/guides/deep-learning-tuning-playbook/faq)

---

# 5. ReduceLROnPlateau

This is different from a predefined schedule.

Instead of saying:

> "At epoch 20, reduce the LR."

we say:

> "If validation performance stops improving, reduce the LR."

Example:

```text
Validation loss

↓
↓
↓
↓
────────────
     plateau
        ↓
   reduce LR
        ↓
     ↓
    ↓
```

For example:

```python
scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)
```

Meaning:

```text
Monitor → validation loss

If it doesn't improve for 3 epochs
        ↓
LR = LR × 0.5
```

---

# 🔥 Learning-Rate Scheduling Mental Model

```text
Initial LR
   ↓
Warmup
   ↓
High learning rate
   ↓
Learning
   ↓
Decay
   ↓
Small learning rate
   ↓
Fine optimization
```

---

# 2️⃣ Early Stopping

## What is Early Stopping?

**Early stopping** automatically stops training when the model stops improving on a validation metric.

This is especially useful for preventing unnecessary training and overfitting.

---

# 🧠 The Problem

Suppose:

```text
Epoch →       1   2   3   4   5   6   7   8

Train loss →  ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓

Val loss →    ↓   ↓   ↓   ↓   ↓   ↑   ↑   ↑
```

Training loss continues decreasing.

But validation loss begins increasing.

That means:

```text
Model is becoming better at training data
BUT
Model is becoming worse at unseen data
```

This is a classic sign of **overfitting**.

---

## 🖼️ Early Stopping Visualization

```markdown
![Early Stopping](https://www.ncbi.nlm.nih.gov/books/NBK597497/bin/515045_1_En_3_Fig8_HTML.jpg)
```

![Early Stopping](https://www.ncbi.nlm.nih.gov/books/NBK597497/bin/515045_1_En_3_Fig8_HTML.jpg)

The figure illustrates the point where validation loss begins increasing and identifies the corresponding model as the preferred stopping point. The source is an openly licensed educational chapter. [NCBI](https://www.ncbi.nlm.nih.gov/books/NBK597497/figure/ch3.Fig8/)

---

# Early Stopping Intuition

Think of training like studying for an exam.

At first:

```text
Study more
   ↓
Knowledge improves
```

But if you keep memorizing the exact practice questions:

```text
Practice performance ↑
Real exam performance ↓
```

You've started **overfitting**.

Early stopping says:

> "Stop when validation performance tells us that further training is no longer helping."

---

# Important Parameters

Typical early-stopping configuration:

```python
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
```

Let's understand each one.

---

## `monitor`

What should we watch?

```python
monitor="val_loss"
```

This means:

> Monitor validation loss.

Could also be:

```python
monitor="val_accuracy"
```

---

# `patience`

Suppose:

```python
patience = 5
```

This means:

> Allow up to 5 epochs without improvement before stopping.

Example:

```text
Epoch 20 → best val_loss
Epoch 21 → worse
Epoch 22 → worse
Epoch 23 → worse
Epoch 24 → worse
Epoch 25 → worse

STOP
```

---

# `restore_best_weights`

This is **very important**.

Suppose:

```text
Epoch 20 → best model
Epoch 21 → worse
Epoch 22 → worse
Epoch 23 → worse
Epoch 24 → worse
Epoch 25 → worse
```

If we simply stop at epoch 25, the current weights might be worse than epoch 20.

With:

```python
restore_best_weights=True
```

the model returns to:

```text
Epoch 20 weights
```

So we keep the best observed validation performance.

---

# Early Stopping Workflow

```text
Train
  ↓
Validate
  ↓
Improvement?
 ┌───────────────┐
 │               │
YES             NO
 │               │
 ↓               ↓
Continue      patience++
                 │
                 ↓
          patience exceeded?
             │        │
            NO       YES
             │        │
             ↓        ↓
          Continue   STOP
```

---

# Early Stopping ≠ Checkpointing

These are related but **not identical**.

### Early Stopping

Answers:

> **When should training stop?**

### Checkpointing

Answers:

> **Which training states should I save?**

You often use both together.

---

# 3️⃣ Checkpointing

## What is Checkpointing?

**Checkpointing** means periodically saving the state of the model during training.

Think of it like a video game:

```text
Start game
    ↓
Save
    ↓
Continue
    ↓
Save
    ↓
Continue
    ↓
Save
```

If something goes wrong:

```text
💥 Training interrupted
        ↓
Load checkpoint
        ↓
Resume
```

Checkpointing is especially important when training takes hours, days, or longer. Keras provides `ModelCheckpoint`, and TensorFlow documents checkpoints as a way to save training progress and resume interrupted training. [Keras](https://keras.io/guides/training_with_built_in_methods/?utm_source=chatgpt.com)

---

# 🖼️ Checkpointing Visualization

```markdown
![Model Checkpointing](https://static.wikidocs.net/images/page/178638/01_Keras_Checkpoints.png)
```

![Model Checkpointing](https://static.wikidocs.net/images/page/178638/01_Keras_Checkpoints.png)

The diagram illustrates saving the model state during training at epoch boundaries. [Wikidocs](https://wikidocs.net/178813)

---

# Why Do We Need Checkpoints?

Imagine training a model for:

```text
72 hours
```

At:

```text
71 hours 45 minutes
```

your machine crashes.

Without checkpointing:

```text
72 hours of work → LOST 😭
```

With checkpointing:

```text
Checkpoint at epoch 60
        ↓
Crash at epoch 72
        ↓
Load epoch 60
        ↓
Continue training
```

Only the work after the last checkpoint needs to be repeated.

---

# What Can a Checkpoint Contain?

A full training checkpoint may contain:

```text
Model weights
      +
Optimizer state
      +
Epoch / step
      +
Learning-rate scheduler state
      +
Other training state
```

This is important because simply saving model weights is not always enough to reproduce the exact training state.

---

# Model Weights vs Training Checkpoint

### Weights

```text
W
b
W
b
...
```

These are the learned parameters.

### Training checkpoint

```text
Model weights
+
Optimizer state
+
Training step
+
Scheduler state
+
Other state
```

Therefore:

```text
Checkpoint > just model weights
```

when your goal is to resume training faithfully.

---

# Best Checkpoint

Suppose:

| Epoch | Validation Loss |
|---:|---:|
| 1 | 0.80 |
| 2 | 0.60 |
| 3 | 0.45 |
| 4 | 0.39 |
| 5 | **0.31** ⭐ |
| 6 | 0.34 |
| 7 | 0.38 |

The best checkpoint is:

```text
Epoch 5
```

not necessarily:

```text
Epoch 7
```

This is why we often save:

```text
save_best_only=True
```

Google's tuning guidance also emphasizes that the **best checkpoint is not necessarily the final checkpoint**, because validation performance can fluctuate or worsen later. [Google for Developers](https://developers.google.com/machine-learning/guides/deep-learning-tuning-playbook/additional-guidance?authuser=0000\&hl=en\&utm_source=chatgpt.com)

---

# Keras Implementation

```python
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min"
)
```

Then:

```python
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    callbacks=[checkpoint]
)
```

Now:

```text
Epoch 1 → save
Epoch 2 → better → save
Epoch 3 → better → save
Epoch 4 → worse → don't replace
Epoch 5 → better → save
```

---

# Checkpointing for Fault Tolerance

Suppose:

```text
Training
   ↓
Epoch 1 ✓
Epoch 2 ✓
Epoch 3 ✓
Epoch 4 ✓
Epoch 5 ✓
💥 CRASH
```

Load:

```text
Epoch 5 checkpoint
```

and resume.

This is called **fault tolerance**.

Keras specifically describes `ModelCheckpoint` as useful for restarting training after interruptions. [Keras](https://keras.io/guides/training_with_built_in_methods/?utm_source=chatgpt.com)

---

# 4️⃣ Gradient Clipping

Now we come to an important optimization-stability technique. 🔥

## The Problem: Exploding Gradients

During backpropagation, gradients can sometimes become extremely large.

For example:

```text
Normal gradient:

0.02
0.04
0.08
```

But an unstable gradient might become:

```text
0.5
2
10
500
10000
```

This can produce enormous parameter updates.

---

# 🧠 Why Is This Dangerous?

Recall:

$$
\theta_{new}
=
\theta_{old}
-
\eta g
$$

where:

$$
g = \nabla_\theta J(\theta)
$$

If:

$$
g \rightarrow \text{very large}
$$

then:

$$
\Delta\theta
=
-\eta g
$$

also becomes very large.

So the optimizer can make a catastrophic jump.

---

# 🖼️ Gradient Clipping

```markdown
![Gradient Clipping](https://developers.google.com/static/machine-learning/guides/deep-learning-tuning-playbook/images/gradient_clipping.png)
```

![Gradient Clipping](https://developers.google.com/static/machine-learning/guides/deep-learning-tuning-playbook/images/gradient_clipping.png)

The Google Deep Learning Tuning Playbook shows how gradient clipping can stabilize training when gradient norms become unusually large. [Google for Developers](https://developers.google.com/machine-learning/guides/deep-learning-tuning-playbook/faq)

---

# What Does Gradient Clipping Do?

Gradient clipping says:

> "If the gradient becomes too large, limit its magnitude before updating the weights."

For example:

```text
Gradient norm = 100

Maximum allowed = 10

             ↓

Gradient norm after clipping = 10
```

---

# Gradient Norm Clipping

Suppose:

$$
\|g\| > \lambda
$$

where:

- $\|g\|$ = gradient norm
- $\lambda$ = clipping threshold

Then:

$$
g'
=
\lambda
\frac{g}{\|g\|}
$$

So the **direction remains the same**, but the magnitude is reduced.

This is the standard norm-clipping formulation described in deep-learning references. [Deep Learning Book](https://www.deeplearningbook.org/contents/rnn?utm_source=chatgpt.com)

---

# 🧠 Visual Intuition

Suppose the gradient vector is:

```text
                 ↑
                 │
                 │
                 │
                 │
                 │
                /
               /
              /
─────────────●────────→
```

The gradient is too large.

Gradient clipping:

```text
Original vector
──────────────────────→

Clipped vector
────────────→
```

We preserve the direction but limit the magnitude.

---

# Gradient Clipping in PyTorch

```python
torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)
```

Typical training:

```python
loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer.step()
```

Notice the order:

```text
Forward
   ↓
Loss
   ↓
Backward
   ↓
Gradients
   ↓
CLIP
   ↓
Optimizer update
```

---

# Gradient Clipping in TensorFlow

TensorFlow/Keras optimizers can also clip gradients.

For example:

```python
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0
)
```

This tells the optimizer to constrain the gradient norm.

---

# Gradient Clipping vs Learning Rate

These are **not the same thing**.

### Learning Rate

Controls the normal size of parameter updates:

$$
\Delta\theta=-\eta g
$$

### Gradient Clipping

Controls unusually large gradients:

```text
Normal gradient
     ↓
No clipping needed

Huge gradient
     ↓
Clip
     ↓
Stable update
```

---

# ⚠️ Don't Clip Too Aggressively

Suppose:

```text
Typical gradient norm = 1
```

and you choose:

```text
max_norm = 0.001
```

Then almost every gradient might be clipped.

That can effectively behave like excessively reducing the update magnitude.

Google's guidance specifically warns that overly aggressive clipping can effectively act like reducing the learning rate. [Google for Developers](https://developers.google.com/machine-learning/guides/deep-learning-tuning-playbook/faq)

So:

```text
Too high threshold
        ↓
May not help

Reasonable threshold
        ↓
Controls outliers

Too low threshold
        ↓
Learning can become unnecessarily slow
```

---

# 🔥 When Is Gradient Clipping Especially Useful?

Gradient clipping is particularly useful when you observe:

- exploding gradients
- sudden loss spikes
- unstable early training
- unstable mid-training behavior
- recurrent/sequence-model training instability

Google's tuning guide notes that clipping can address both early and mid-training instability caused by unusually large gradient norms. [Google for Developers](https://developers.google.com/machine-learning/guides/deep-learning-tuning-playbook/faq)

---

# 🧠 Training Techniques — Big Picture

Now connect all four:

```text
                 TRAINING
                    │
       ┌────────────┼────────────┐
       ↓            ↓            ↓
Learning Rate   Validation   Checkpoint
  Schedule       Monitoring      Saving
       │            │            │
       ↓            ↓            ↓
   Optimize     Early Stop    Recovery
       │
       ↓
Gradient Clipping
       │
       ↓
Stable Optimization
```

---

# 5️⃣ Experiment Tracking

Now comes a very important **industry skill**. 🔥

Imagine you run:

```text
Experiment 1
Experiment 2
Experiment 3
...
Experiment 50
```

And each experiment has:

```text
Learning rate
Batch size
Optimizer
Dropout
Weight decay
Model architecture
Training loss
Validation loss
Accuracy
Training time
```

After 50 experiments, you might ask:

> "Which model was actually the best?"

If you didn't record anything properly:

😭 **Good luck remembering.**

That's why we use **experiment tracking**.

---

# What Is Experiment Tracking?

Experiment tracking means systematically recording:

```text
Configuration
     +
Metrics
     +
Model information
     +
Training information
     +
Results
```

For every experiment.

---

# Example

### Experiment 001

```text
Model: ResNet
Optimizer: Adam
LR: 0.001
Batch: 32
Dropout: 0.2

Train accuracy: 95%
Val accuracy: 91%
```

### Experiment 002

```text
Model: ResNet
Optimizer: AdamW
LR: 0.0005
Batch: 32
Dropout: 0.2

Train accuracy: 94%
Val accuracy: 93%
```

Now we can compare them.

---

# Why Experiment Tracking Matters

Without tracking:

```text
Train
 ↓
Change something
 ↓
Train
 ↓
Change something
 ↓
Train
 ↓
🤔 What actually worked?
```

With tracking:

```text
Experiment
    ↓
Log
    ↓
Compare
    ↓
Analyze
    ↓
Best configuration
```

This is fundamental to reproducible ML experimentation.

---

# 6️⃣ TensorBoard

## What is TensorBoard?

**TensorBoard** is a visualization and experimentation tool commonly used with TensorFlow.

It allows us to visualize things such as:

- loss
- accuracy
- learning rate
- model graphs
- weights
- biases
- histograms
- images
- embeddings
- profiling information

TensorFlow describes TensorBoard as a suite of visualization tools for understanding, debugging, and optimizing ML programs. [TensorFlow](https://www.tensorflow.org/tensorboard?utm_source=chatgpt.com)

---

# 🖼️ TensorBoard Dashboard

```markdown
![TensorBoard Training Dashboard](https://docs.pytorch.org/tutorials/intermediate/_static/img/tensorboard_scalar_runs.png)
```

![TensorBoard Training Dashboard](https://docs.pytorch.org/tutorials/intermediate/_static/img/tensorboard_scalar_runs.png)

The TensorBoard tutorial demonstrates using the Scalars dashboard to visualize training metrics such as loss over iterations. [PyTorch Documentation](https://docs.pytorch.org/tutorials/intermediate/tensorboard_tutorial.html?highlight=no_grad\&utm_source=chatgpt.com)

---

# TensorBoard Mental Model

```text
Training Code
     │
     │ logs metrics
     ↓
Log Directory
     │
     ↓
TensorBoard
     │
 ┌───┼───────────┐
 ↓   ↓           ↓
Loss Accuracy  LR
 ↓
Graphs
```

---

# What Can We Track?

## Loss

```text
Epoch
  ↓
Loss
  ↓
↓
↓
↓
```

Ideally, training loss decreases.

---

## Accuracy

```text
Epoch
  ↓
Accuracy
  ↑
↑
↑
```

Ideally, useful performance increases.

---

## Learning Rate

We can visualize whether our schedule is behaving as intended.

TensorFlow's TensorBoard examples explicitly show tracking loss, accuracy, and learning rate over training. [TensorFlow](https://www.tensorflow.org/tensorboard/get_started?authuser=956858896\&utm_source=chatgpt.com)

---

# Training vs Validation

One of the most useful things to visualize is:

```text
Training loss
vs
Validation loss
```

For example:

```text
Loss
│\
│ \
│  \
│   \________ Training
│
│   \__
│      \__
│         \____ Validation
│              /
│             /
└──────────────── Epoch
```

If:

```text
Train loss ↓
Val loss ↑
```

we should investigate overfitting.

---

# TensorBoard with Keras

Basic implementation:

```python
import tensorflow as tf
from tensorflow import keras

log_dir = "logs"

tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir
)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    callbacks=[tensorboard_callback]
)
```

Then launch:

```python
%load_ext tensorboard
%tensorboard --logdir logs
```

TensorFlow documents the `%tensorboard` notebook workflow for launching TensorBoard and monitoring training. [TensorFlow](https://www.tensorflow.org/tensorboard/get_started?authuser=956858896\&utm_source=chatgpt.com)

---

# TensorBoard Important Dashboards

## Scalars

Shows:

```text
Loss
Accuracy
Learning rate
Other numerical metrics
```

---

## Graphs

Shows the computational/model graph.

Useful for checking:

```text
Input
 ↓
Layer
 ↓
Layer
 ↓
Output
```

TensorFlow's Graphs dashboard is designed to inspect the model structure and verify that the graph matches the intended design. [TensorFlow](https://www.tensorflow.org/tensorboard/graphs?utm_source=chatgpt.com)

---

## Histograms

Can visualize distributions of:

```text
Weights
Biases
Activations
Other tensors
```

---

## Images

Can display:

```text
Input images
Generated images
Predictions
Confusion matrices
```

TensorBoard's image summaries can log images and inspect them throughout training. [TensorFlow](https://www.tensorflow.org/tensorboard/image_summaries?utm_source=chatgpt.com)

---

# TensorBoard for Experiment Comparison

Suppose:

```text
Run 1 → Adam
Run 2 → AdamW
Run 3 → SGD
Run 4 → Adam + dropout
```

TensorBoard can let us compare their curves.

This is much better than manually writing:

```text
Experiment 1 = 91.2%
Experiment 2 = 92.1%
...
```

---

# 7️⃣ Weights & Biases — W&B

Your syllabus marks **Weights & Biases as optional**, so you don't need to make this as deep as TensorBoard. syllabus

But you should understand what it is.

---

# What is W&B?

**Weights & Biases (W&B)** is an experiment-tracking platform for machine learning.

It can track:

```text
Experiments
Hyperparameters
Metrics
Model versions
Artifacts
System metrics
Visualizations
```

W&B describes its experiment system as a way to track models, metrics, and hyperparameters and compare experiments through dashboards. [Weights & Biases](https://wandb.ai/site/experiment-tracking/)

---

# 🖼️ W&B Dashboard

```markdown
![Weights and Biases Experiment Dashboard](https://site.wandb.ai/wp-content/uploads/2023/07/60c2531e5144f46276377517_visualize-metrics-p-1600.png?w=800)
```

![Weights and Biases Experiment Dashboard](https://site.wandb.ai/wp-content/uploads/2023/07/60c2531e5144f46276377517_visualize-metrics-p-1600.png?w=800)

This W&B dashboard demonstrates comparing multiple experiment runs and visualizing metrics such as accuracy, loss, GPU utilization, and validation accuracy. [Weights & Biases](https://site.wandb.ai/wp-content/uploads/2023/07/60c2531e5144f46276377517_visualize-metrics-p-1600.png?w=800)

---

# W&B Basic Workflow

```text
Start Run
   ↓
Log Hyperparameters
   ↓
Train Model
   ↓
Log Metrics
   ↓
Save Artifacts
   ↓
Compare Runs
```

---

# Basic W&B Example

```python
import wandb

run = wandb.init(
    project="deep-learning-experiments"
)

wandb.config.learning_rate = 0.001
wandb.config.batch_size = 32
wandb.config.dropout = 0.2
```

During training:

```python
wandb.log({
    "loss": loss,
    "accuracy": accuracy,
    "val_loss": val_loss,
    "val_accuracy": val_accuracy
})
```

W&B's current experiment-tracking documentation uses the same basic pattern: initialize a run, record configuration, and log metrics over time. [Weights & Biases](https://wandb.ai/site/experiment-tracking/)

---

# TensorBoard vs W&B

| Feature | TensorBoard | W&B |
|---|---|---|
| Training metrics | ✅ | ✅ |
| Loss curves | ✅ | ✅ |
| Accuracy curves | ✅ | ✅ |
| Learning-rate tracking | ✅ | ✅ |
| Model graph | ✅ | ✅ |
| Histograms | ✅ | ✅ |
| Image logging | ✅ | ✅ |
| Experiment comparison | ✅ | ✅ |
| Cloud dashboard | Mainly local/self-managed | ✅ Strong |
| Team collaboration | Basic | ✅ Strong |
| Artifact/model tracking | Limited compared with W&B | ✅ |
| Hyperparameter sweeps | Available through plugins/tools | ✅ Strong |
| Learning priority | **Must know** | Optional in your syllabus |

---

# 🧠 TensorBoard vs W&B — Simple Understanding

Think:

### TensorBoard

> **"I want to see what my model is doing during training."**

### W&B

> **"I want to manage, compare, reproduce, and collaborate on many experiments."**

Both can overlap heavily.

---

# 🔥 Complete Training Pipeline

Now put everything together.

```text
                 DATASET
                    ↓
              MODEL SETUP
                    ↓
            Hyperparameters
                    ↓
              TRAIN MODEL
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
     Scheduler   Gradient    Checkpoint
                  Clipping
        │           │           │
        └───────────┼───────────┘
                    ↓
                Validation
                    ↓
             Early Stopping
                    ↓
              Experiment Logs
                    ↓
        ┌───────────┴───────────┐
        ↓                       ↓
   TensorBoard                 W&B
        ↓                       ↓
    Visualize                Track
        │                       │
        └───────────┬───────────┘
                    ↓
             Best Experiment
                    ↓
              Best Checkpoint
                    ↓
              Final Evaluation
```

---

# 🔥 How These Techniques Work Together

Suppose we train a CNN.

Initial setup:

```text
Optimizer = AdamW
Learning rate = 0.001
Batch size = 32
Weight decay = 0.0001
Dropout = 0.2
```

Then:

### Step 1 — Warmup

Start with:

```text
LR = 0.00001
```

Gradually increase.

---

### Step 2 — Normal Training

```text
LR = 0.001
```

Model learns rapidly.

---

### Step 3 — LR Scheduling

As training progresses:

```text
0.001
 ↓
0.0005
 ↓
0.0001
 ↓
0.00001
```

---

### Step 4 — Gradient Clipping

If a huge gradient occurs:

```text
Gradient norm = 20

max_norm = 1

       ↓

Gradient clipped
```

Training remains stable.

---

### Step 5 — Checkpointing

Every epoch:

```text
Save checkpoint
```

If validation improves:

```text
Save as best checkpoint ⭐
```

---

### Step 6 — Early Stopping

Suppose:

```text
val_loss

0.50
0.40
0.35
0.31 ⭐
0.32
0.33
0.34
0.35
```

After sufficient patience:

```text
STOP
```

Then:

```text
Restore best checkpoint
```

---

### Step 7 — Experiment Tracking

TensorBoard/W&B records:

```text
Learning rate
Loss
Validation loss
Accuracy
Validation accuracy
Training time
Experiment configuration
```

Now you can explain **why your final model is good**.

---

# 🧪 REAL EXPERIMENT

Suppose we want to determine whether learning-rate scheduling improves our model.

## Experiment A — Constant LR

```text
Optimizer = Adam
LR = 0.001
Scheduler = None
```

Results:

```text
Validation Accuracy = 91.2%
```

---

## Experiment B — Step Decay

```text
Optimizer = Adam
Initial LR = 0.001
Scheduler = Step Decay
```

Results:

```text
Validation Accuracy = 92.4%
```

---

## Experiment C — Cosine Annealing

```text
Optimizer = Adam
Initial LR = 0.001
Scheduler = Cosine
```

Results:

```text
Validation Accuracy = 93.1%
```

Now you have evidence:

```text
Constant LR
    ↓
91.2%

Step Decay
    ↓
92.4%

Cosine
    ↓
93.1% ⭐
```

That's **actual experimentation**.

---

# 🚨 One Important Rule

Never conclude:

> "Cosine annealing is better."

just because one experiment got a higher score.

You should control:

```text
Dataset
Model architecture
Random seed
Batch size
Optimizer
Training budget
Evaluation procedure
```

and change the intended variable.

Then compare.

---

# 🧠 The Four Training Techniques

Memorize this:

| Technique | Main Question |
|---|---|
| Learning-rate scheduling | **How should the step size change?** |
| Early stopping | **When should training stop?** |
| Checkpointing | **What training states should I save?** |
| Gradient clipping | **How do I prevent extreme gradient updates?** |

---

# 🧠 Experiment Tracking

| Tool | Main Purpose |
|---|---|
| TensorBoard | Visualize and inspect training |
| W&B | Track, compare, reproduce, and collaborate across experiments |

---

# 🔥 Industry Mental Model

When training a serious model:

```text
                    MODEL
                      │
                      ↓
                TRAINING LOOP
                      │
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
      LR            Gradients     Metrics
   Scheduler        Clipping         │
        │             │              ↓
        └─────────────┼────────── TensorBoard
                      │
                      ↓
                 Checkpoints
                      │
                      ↓
               Validation
                      │
                      ↓
              Early Stopping
                      │
                      ↓
              Best Checkpoint
                      │
                      ↓
              Experiment Result
```

This is the kind of workflow you should be comfortable explaining in an **AI/ML Engineer interview**. 🚀

---

# 🎯 Quick Interview Questions

### Q1. Why use a learning-rate scheduler?

To change the learning rate during training so that optimization can make large progress early and finer adjustments later.

---

### Q2. What is early stopping?

A technique that stops training when a monitored validation metric stops improving for a specified patience period.

---

### Q3. Why use checkpointing?

To save training state so that we can recover from interruptions and/or select the best-performing model checkpoint.

---

### Q4. What problem does gradient clipping solve?

It limits unusually large gradients and helps stabilize training when gradient explosions or large gradient spikes occur.

---

### Q5. What is the difference between early stopping and checkpointing?

```text
Early stopping
→ decides WHEN to stop.

Checkpointing
→ decides WHAT training state to save.
```

---

### Q6. What does TensorBoard do?

It visualizes and helps monitor training metrics, model graphs, distributions, images, and other experiment information. [TensorFlow](https://www.tensorflow.org/tensorboard?utm_source=chatgpt.com)

---

### Q7. Why is experiment tracking important?

Because it allows us to:

- reproduce experiments
- compare configurations
- understand what changed
- identify the best configuration
- maintain a record of model development

---

# 🧠 FINAL MASTER SUMMARY

```text
                    DEEP LEARNING EXPERIMENTATION
                              │
          ┌───────────────────┴───────────────────┐
          │                                       │
    HYPERPARAMETERS                         TRAINING
          │                                       │
          │                         ┌─────────────┼─────────────┐
          │                         │             │             │
     Learning Rate              Scheduling    Early Stop   Checkpoint
     Batch Size                     │             │             │
     Hidden Units                   │             │             │
     Layers                         └──────┬──────┴──────┬──────┘
     Dropout                               │             │
     Weight Decay                    Gradient Clip      │
     Optimizer                            │             │
          │                              └──────┬──────┘
          └─────────────────────────────────────┤
                                                ↓
                                      EXPERIMENT TRACKING
                                                │
                                      ┌─────────┴─────────┐
                                      ↓                   ↓
                                 TensorBoard             W&B
                                      │                   │
                                      └─────────┬─────────┘
                                                ↓
                                      Compare Experiments
                                                ↓
                                        Best Configuration
                                                ↓
                                          Best Checkpoint
```

---

# 🔥 The Most Important Concept

Don't think of these as isolated techniques.

Think of them as **one complete training system**:

> **Learning-rate scheduling controls how aggressively the model learns.**

> **Gradient clipping protects the optimization process from extreme gradients.**

> **Checkpointing protects your training work and stores good model states.**

> **Early stopping prevents unnecessary/overfit training.**

> **TensorBoard/W&B records what happened so you can analyze and reproduce your experiments.**

That's the real meaning of **Deep Learning Experimentation**. ❤️‍🔥

---

# 📝 Practical Keras Training Setup

Here's what all of this can look like together:

```python
import tensorflow as tf
from tensorflow import keras

# -------------------------
# Callbacks
# -------------------------

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min"
)

tensorboard = keras.callbacks.TensorBoard(
    log_dir="logs"
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# -------------------------
# Optimizer
# -------------------------

optimizer = keras.optimizers.AdamW(
    learning_rate=0.001,
    weight_decay=1e-4,
    clipnorm=1.0
)

# -------------------------
# Compile
# -------------------------

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# -------------------------
# Train
# -------------------------

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[
        early_stopping,
        checkpoint,
        tensorboard,
        reduce_lr
    ]
)
```

Now you have:

```text
AdamW
 +
Weight decay
 +
Gradient clipping
 +
LR reduction
 +
Early stopping
 +
Checkpointing
 +
TensorBoard
```

---